BLOCK 1 — Import Libraries and Load Data

In [0]:
# ------------------------------------------------------------
# Block 1: Load Sampled Dataset (NO FULL SILVER)
# ------------------------------------------------------------

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.table("workspace.silver.labeled_step_test_sampled")

row_count = df.count()
print(f"Loaded row count: {row_count}")

assert row_count == 206201, "WRONG DATASET LOADED – STOP HERE"

BLOCK 2 — Logistic Regression Hyperparameter Tuning

In [0]:
# ------------------------------------------------------------
# Block 2: Load Training Artifacts
# ------------------------------------------------------------

import joblib

ARTIFACT_DIR = "/Workspace/Repos/win185@ensign.edu/Databricks/etl"

X_train = joblib.load(f"{ARTIFACT_DIR}/X_train_transformed.pkl")
X_test  = joblib.load(f"{ARTIFACT_DIR}/X_test_transformed.pkl")
y_train = joblib.load(f"{ARTIFACT_DIR}/y_train.pkl")
y_test  = joblib.load(f"{ARTIFACT_DIR}/y_test.pkl")

print(
    X_train.shape,
    X_test.shape,
    y_train.shape,
    y_test.shape
)

BLOCK 3 — Random Forest Hyperparameter Tuning

In [0]:
# ------------------------------------------------------------
# Model Training & Evaluation (FINAL)
# ------------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# DROP distance-based feature (index 0)
# Keep only sensor_type one-hot features
X_train_reduced = X_train[:, 1:]
X_test_reduced  = X_test[:, 1:]

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_reduced, y_train)

y_pred = rf.predict(X_test_reduced)

print("Test accuracy:", accuracy_score(y_test, y_pred))

BLOCK 4 — Compare Tuned Models

In [0]:
# ------------------------------------------------------------
# Compare Tuned Models
# ------------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# Model A: Tuned model WITH distance_cm (leakage-prone baseline)
# ------------------------------------------------------------
model_with_distance = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

model_with_distance.fit(X_train, y_train)
y_pred_with_distance = model_with_distance.predict(X_test)

acc_with_distance = accuracy_score(y_test, y_pred_with_distance)

# ------------------------------------------------------------
# Model B: Tuned model WITHOUT distance_cm (leakage-safe)
# ------------------------------------------------------------
X_train_reduced = X_train[:, 1:]
X_test_reduced  = X_test[:, 1:]

model_without_distance = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

model_without_distance.fit(X_train_reduced, y_train)
y_pred_without_distance = model_without_distance.predict(X_test_reduced)

acc_without_distance = accuracy_score(y_test, y_pred_without_distance)

# ------------------------------------------------------------
# Comparison Output
# ------------------------------------------------------------
print("Model comparison results:")
print(f"With distance_cm feature    : {acc_with_distance}")
print(f"Without distance_cm feature : {acc_without_distance}")

BLOCK 5 — Choose Best Model and Save It

In [0]:
# ------------------------------------------------------------
# Select Best Model & Persist
# ------------------------------------------------------------

import joblib
import os

# Select the leakage-safe model as the final model
final_model = model_without_distance
final_accuracy = acc_without_distance

print("Selected final model: RandomForest (without distance_cm)")
print("Final test accuracy:", final_accuracy)

# Save final model to GitHub repo
ARTIFACT_DIR = "/Workspace/Repos/win185@ensign.edu/Databricks/etl"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

joblib.dump(final_model, f"{ARTIFACT_DIR}/final_random_forest_model.pkl")

print("Final model saved to:")
print(f"{ARTIFACT_DIR}/final_random_forest_model.pkl")

Step 6: Evaluation & Ethics Reflection

Model Evaluation Report

In this project, multiple classification models were evaluated to predict step activity using sensor data. Initial model training revealed unrealistically high performance scores, including perfect accuracy, which prompted further investigation. This issue was traced to feature leakage, where the target label was indirectly derived from the same feature (distance_cm) that was also being used as a model input.

To address this, the modeling approach was revised to remove the leaked feature from the training data. Two tuned Random Forest models were then compared: one trained with the full feature set (including distance_cm) and one trained with a reduced feature set that excluded this distance-based signal. The reduced-feature model produced a lower but more realistic test accuracy, reflecting true generalization rather than memorization.

Final evaluation was therefore based on the leakage-safe model, which provides a more honest assessment of predictive performance. This comparison demonstrated the importance of careful feature selection and validation when evaluating machine learning models.


Ethics and Fairness Reflection

This project highlights an important ethical consideration in machine learning related to label construction and data leakage. Initially, the target variable was created using a quantile-based threshold on distance_cm. When this same feature was also included in model training, it resulted in artificially high accuracy scores that did not reflect genuine learning.

From an ethical and fairness perspective, reporting such inflated performance without identifying the source would be misleading and could result in unjustified confidence in the model’s predictive ability. This is particularly concerning in real-world applications, where decisions based on overfit or leaked models could negatively impact individuals or groups.

Rather than accepting these misleading results, the workflow explicitly identified and corrected the leakage by removing the distance-based feature during model training and comparing model performance before and after the correction. This approach prioritizes transparency, methodological rigor, and responsible reporting over maximizing performance metrics.

Future improvements should focus on designing labels that are independent of input features and on validating datasets to ensure that model performance reflects true predictive capability. These steps are essential for building fair, trustworthy, and ethically sound machine learning systems.